# 03 — 3D tumor segmentation with MONAI (new)

Input: FLAIR, T1, T1ce, T2. Output: the 3 BraTS regions (TC, WT, ET) -> labels 0/1/2/4.
Models: `unet3d` (3D U-Net), `attention_unet`, `segresnet`, `swin_unetr`.
Metrics: Dice, IoU, HD95 per region on the test split -> `results/eval/seg_<model>.json` (read by the app).

**Runtime → Change runtime type → T4 GPU.** About 1-2 min per epoch on a T4. Checkpoints are written to
Drive after every epoch, so you can resume if Colab disconnects.

In [ ]:
# --- Setup: mount Drive + install the BrainTumorLab code (run this cell first) ---
from google.colab import drive
drive.mount("/content/drive")

import os, sys
REPO_URL = ""   # e.g. "https://github.com/<your-account>/BrainTumorLab.git" ; empty = use the zip on Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/miniProjetAI/BrainTumorLab.zip"
if not os.path.isfile("/content/BrainTumorLab/pyproject.toml"):
    import shutil
    shutil.rmtree("/content/BrainTumorLab", ignore_errors=True)
    if REPO_URL:
        !git clone -q {REPO_URL} /content/BrainTumorLab
    else:
        !rm -rf /content/btl_tmp && unzip -q -o "{ZIP_ON_DRIVE}" -d /content/btl_tmp
        _src = "/content/btl_tmp/BrainTumorLab" if os.path.isfile("/content/btl_tmp/BrainTumorLab/pyproject.toml") else "/content/btl_tmp"
        shutil.move(_src, "/content/BrainTumorLab")
!pip install -q -e /content/BrainTumorLab
sys.path.insert(0, "/content/BrainTumorLab/src")

import torch
from braintumorlab.config import Paths, seed_everything
seed_everything(0)
paths = Paths("/content/drive/MyDrive/miniProjetAI")
DATA_ROOT = paths.ensure_data_extracted()     # unzips BraTS_2020_Data_preprocessing_nii_FIXED.zip once
print("data:", DATA_ROOT, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
from braintumorlab.segmentation import SegConfig, prepare_cache, train_segmentation, evaluate_model
cfg = SegConfig(data_root=str(DATA_ROOT), cache_dir="/content/seg_cache",
                out_dir=str(paths.seg_models), model_name="unet3d",   # or "segresnet", "attention_unet"
                roi=(128, 128, 128), epochs=60, batch_size=1, val_every=2, max_val=20)
prepare_cache(cfg)          # float16 copies on the local disk (once per Colab session, ~5 min)

In [ ]:
best, hist = train_segmentation(cfg)
# resume after a disconnection:
# best, hist = train_segmentation(cfg, resume=f"{cfg.out_dir}/{cfg.model_name}/last.pt")

In [ ]:
import pandas as pd
h = pd.DataFrame(hist)
h.plot(x="epoch", y="train_loss"); h.dropna().plot(x="epoch", y=["val_dice_WT", "val_dice_TC", "val_dice_ET"])

In [ ]:
result = evaluate_model(f"{cfg.out_dir}/{cfg.model_name}/best.pt", cfg, split="test", eval_dir=str(paths.eval_dir))
pd.DataFrame({r: {k: v["mean"] for k, v in result["summary"][r].items() if v} for r in ("WT", "TC", "ET")}).round(3)

### Compare models
Change `cfg.model_name` (e.g. `"segresnet"`), re-run training and evaluation: every model gets its own
JSON file and appears in the **Model Comparison** page of the app.